In [1]:
# =============================================================================
# CHAPTER 4.5 — EXTERNAL VALIDATION: STUFFMART CUSTOMER CONVERSION DATASET
#
# This notebook implements the external robustness check described in
# Chapter 3 Section 3.2.1 of the dissertation.
#
# PURPOSE:
#   Test whether the directional finding from X Education holds in a
#   structurally different conversion context:
#   Does S1 (ML only) outperform S3_uncapped (hard priority without cap)?
#
# SCOPE:
#   Three systems only: S1, S2, S3 (capped and uncapped)
#   No S4 soft adjustment — not implemented for secondary dataset
#   No bootstrap — this is directional robustness checking, not inference
#   Results are compared directionally to X Education primary findings
#
# DATASET:
#   Stuffmart Customer Conversion Dataset (Kaggle)
#   100,000 synthetic leads | ~1.65% conversion rate | Pakistan retail context
#   Two CSVs: training (customer_conversion_traing_dataset.csv)
#             testing  (customer_conversion_testing_dataset.csv)
#
# UPLOAD INSTRUCTIONS:
#   Upload both CSVs to your Google Drive under:
#   /content/drive/MyDrive/Masters_Constrained_Lead_Qualification/
#   Then mount Drive in Cell SM-00.
#
# IMPORTANT — HARDCODED X EDUCATION NUMBERS:
#   Cell SM-10d contains manually entered X Education results for the
#   cross-dataset comparison chart. These are taken from the final
#   primary notebook (Notebook 15). If the primary results change,
#   update XE_PREC and XE_BASELINE in Cell SM-10d accordingly.
#   They are clearly labelled to make this audit trail transparent.
# =============================================================================



In [2]:

# =============================================================================
# CELL SM-00 — SETUP AND MOUNT DRIVE
# =============================================================================

!pip install xgboost -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/Masters_Constrained_Lead_Qualification/'
TRAIN_FILE = DRIVE_PATH + 'customer_conversion_traing_dataset .csv'
TEST_FILE  = DRIVE_PATH + 'customer_conversion_testing_dataset.csv'

print("Setup complete.")



Mounted at /content/drive
Setup complete.


In [3]:

# =============================================================================
# CELL SM-01 — LOAD DATA
# =============================================================================

sm_train_raw = pd.read_csv(TRAIN_FILE)
sm_test_raw  = pd.read_csv(TEST_FILE)

print(f"Training set shape: {sm_train_raw.shape}")
print(f"Test set shape:     {sm_test_raw.shape}")
print(f"\nColumns: {sm_train_raw.columns.tolist()}")
print(f"\nTraining conversion rate: {sm_train_raw['Conversion (Target)'].mean():.4f}")
print(f"Test conversion rate:     {sm_test_raw['Conversion (Target)'].mean():.4f}")
print(f"\nLeadStatus values (train):\n{sm_train_raw['LeadStatus'].value_counts()}")
print(f"\nLeadSource values (train):\n{sm_train_raw['LeadSource'].value_counts()}")
print(f"\nPaymentHistory values (train):\n{sm_train_raw['PaymentHistory'].value_counts()}")



Training set shape: (100000, 19)
Test set shape:     (26145, 19)

Columns: ['LeadID', 'Age', 'Gender', 'Location', 'LeadSource', 'TimeSpent (minutes)', 'PagesViewed', 'LeadStatus', 'EmailSent', 'DeviceType', 'ReferralSource', 'FormSubmissions', 'Downloads', 'CTR_ProductPage', 'ResponseTime (hours)', 'FollowUpEmails', 'SocialMediaEngagement', 'PaymentHistory', 'Conversion (Target)']

Training conversion rate: 0.0165
Test conversion rate:     0.0158

LeadStatus values (train):
LeadStatus
Cold    33435
Hot     33288
Warm    33277
Name: count, dtype: int64

LeadSource values (train):
LeadSource
Organic         25257
Social Media    25030
Email           24947
Referral        24766
Name: count, dtype: int64

PaymentHistory values (train):
PaymentHistory
Good          50111
No Payment    49889
Name: count, dtype: int64
